# Retail Orders: Data Profiling & Quality Checks

RabTech Academy internship deliverable. This notebook profiles the provided retail orders CSV, executes data-quality checks, and exports a results report.

**Important:** checks flag problems; they do not silently repair source data. Resolve issues with the data owner before publishing official KPIs.


## 1. Upload the dataset
Run this cell in Google Colab and upload `retail-orders-raw.csv`. If the file is already in the Colab session, you can skip the upload and set `FILE_PATH` to its location.

In [1]:
from google.colab import files
uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))


Saving retail-orders-raw.csv to retail-orders-raw.csv
Uploaded files: ['retail-orders-raw.csv']


In [2]:
import pandas as pd
import numpy as np
from datetime import date
from pathlib import Path

FILE_PATH = 'retail-orders-raw.csv'  # Change this if your uploaded filename differs
df_raw = pd.read_csv(FILE_PATH, dtype={'order_id': 'string', 'quantity': 'string'})
print('Rows, columns:', df_raw.shape)
display(df_raw.head())
display(pd.DataFrame({'column': df_raw.columns, 'dtype': df_raw.dtypes.astype(str).values, 'missing_count': df_raw.isna().sum().values}))


Rows, columns: (12, 9)


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid


,column,dtype,missing_count
0,order_id,string,0
1,order_date,object,1
2,customer_segment,object,0
3,city,object,1
4,category,object,0
5,quantity,string,0
6,unit_price,int64,0
7,discount_pct,float64,1
8,payment_status,object,0


## 2. Data profile
Review missing values, duplicate rows, duplicate order IDs, and raw value distributions.

In [3]:
profile = pd.DataFrame({
    'column': df_raw.columns,
    'dtype': [str(df_raw[c].dtype) for c in df_raw.columns],
    'non_null_count': [int(df_raw[c].notna().sum()) for c in df_raw.columns],
    'missing_count': [int(df_raw[c].isna().sum()) for c in df_raw.columns],
    'missing_pct': [round(float(df_raw[c].isna().mean() * 100), 2) for c in df_raw.columns],
    'distinct_values': [int(df_raw[c].nunique(dropna=True)) for c in df_raw.columns]
})
display(profile)
print('Exact duplicate rows:', int(df_raw.duplicated().sum()))
print('Duplicate order IDs (all rows with repeated ID):')
display(df_raw[df_raw['order_id'].duplicated(keep=False)].sort_values('order_id'))
for col in ['customer_segment', 'category', 'payment_status']:
    print(f'\\nValues in {col}:')
    display(df_raw[col].value_counts(dropna=False).rename_axis(col).reset_index(name='count'))


,column,dtype,non_null_count,missing_count,missing_pct,distinct_values
0,order_id,string,12,0,0.00,11
1,order_date,object,11,1,8.33,10
2,customer_segment,object,12,0,0.00,4
3,city,object,11,1,8.33,7
4,category,object,12,0,0.00,3
5,quantity,string,12,0,0.00,5
6,unit_price,int64,12,0,0.00,3
7,discount_pct,float64,11,1,8.33,5
8,payment_status,object,12,0,0.00,5


Exact duplicate rows: 1
Duplicate order IDs (all rows with repeated ID):


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid


\nValues in customer_segment:


,customer_segment,count
0,Student,4
1,Professional,4
2,Fresher,3
3,student,1


\nValues in category:


,category,count
0,Learning Kit,5
1,Course Access,4
2,Mentor Session,3


\nValues in payment_status:


,payment_status,count
0,Paid,7
1,Pending,2
2,paid,1
3,Failed,1
4,Refunded,1


## 3. Executable data-quality checks
The source dictionary says dates must be ISO dates from 2025-01-01 through today; order IDs are required and unique; quantity is a whole number greater than zero; unit price is non-negative; discount is between 0 and 100; and categories/statuses must map to approved values.

In [4]:
today = pd.Timestamp(date.today())
checks = []

def add_check(dimension, column, issue, failed_count, total_count, expected_rule, action):
    checks.append({
        'dimension': dimension, 'column': column, 'issue': issue,
        'failed_count': int(failed_count), 'total_count': int(total_count),
        'status': 'PASS' if int(failed_count) == 0 else 'FAIL',
        'expected_rule': expected_rule, 'recommended_action': action
    })

# Completeness
for col in ['order_id', 'order_date', 'customer_segment', 'city', 'category', 'quantity', 'unit_price', 'payment_status']:
    add_check('Completeness', col, 'Required value missing', df_raw[col].isna().sum(), len(df_raw), 'Required fields must be populated', 'Recover from source; do not invent values.')
add_check('Completeness', 'discount_pct', 'Discount missing', df_raw['discount_pct'].isna().sum(), len(df_raw), 'Missing discount needs business justification', 'Do not assume zero without approval.')

# Uniqueness
dup_order_mask = df_raw['order_id'].duplicated(keep=False) | df_raw['order_id'].isna()
add_check('Uniqueness', 'order_id', 'Order ID duplicated or missing', dup_order_mask.sum(), len(df_raw), 'Order ID required and unique', 'Investigate repeated identifiers with the source owner.')

# Strict ISO date validation: a non-ISO representation is flagged even if parseable.
date_text = df_raw['order_date'].astype('string')
iso_shape = date_text.str.fullmatch(r'\d{4}-\d{2}-\d{2}', na=False)
parsed_date = pd.to_datetime(date_text.where(iso_shape), format='%Y-%m-%d', errors='coerce')
bad_date = parsed_date.isna() | (parsed_date < pd.Timestamp('2025-01-01')) | (parsed_date > today)
add_check('Validity', 'order_date', 'Missing, non-ISO, invalid, or out-of-range date', bad_date.sum(), len(df_raw), 'ISO date from 2025-01-01 through today', 'Confirm and correct dates from source.')

# Quantity must be a positive integer.
quantity_num = pd.to_numeric(df_raw['quantity'], errors='coerce')
bad_qty = quantity_num.isna() | (quantity_num <= 0) | (quantity_num % 1 != 0)
add_check('Validity', 'quantity', 'Quantity not a positive whole number', bad_qty.sum(), len(df_raw), 'Whole number greater than zero', 'Correct malformed or non-positive quantities at source.')

price_num = pd.to_numeric(df_raw['unit_price'], errors='coerce')
bad_price = price_num.isna() | (price_num < 0)
add_check('Validity', 'unit_price', 'Unit price missing, non-numeric, or negative', bad_price.sum(), len(df_raw), 'Non-negative INR amount', 'Validate against product/pricing source.')

discount_num = pd.to_numeric(df_raw['discount_pct'], errors='coerce')
bad_discount = discount_num.isna() | (discount_num < 0) | (discount_num > 100)
add_check('Validity', 'discount_pct', 'Discount missing or outside 0–100', bad_discount.sum(), len(df_raw), 'Between 0 and 100; missing needs justification', 'Resolve missing/out-of-range values with business owner.')

allowed_segments = {'student', 'fresher', 'professional'}
segment_norm = df_raw['customer_segment'].astype('string').str.strip().str.casefold()
bad_segment = ~segment_norm.isin(allowed_segments) | segment_norm.isna()
add_check('Consistency', 'customer_segment', 'Segment outside approved values after normalization', bad_segment.sum(), len(df_raw), 'Student, Fresher, Professional after normalization', 'Normalize case/whitespace and investigate unexpected values.')

allowed_categories = {'learning kit', 'course access', 'mentor session'}
category_norm = df_raw['category'].astype('string').str.strip().str.casefold()
bad_category = ~category_norm.isin(allowed_categories) | category_norm.isna()
add_check('Consistency', 'category', 'Category outside approved values after normalization', bad_category.sum(), len(df_raw), 'Learning Kit, Course Access, Mentor Session', 'Map only known equivalent values; investigate others.')

allowed_status = {'paid', 'pending', 'failed', 'refunded'}
status_norm = df_raw['payment_status'].astype('string').str.strip().str.casefold()
bad_status = ~status_norm.isin(allowed_status) | status_norm.isna()
add_check('Consistency', 'payment_status', 'Payment status outside approved values after normalization', bad_status.sum(), len(df_raw), 'Paid, Pending, Failed, Refunded', 'Normalize case/whitespace and investigate unexpected values.')

quality_report = pd.DataFrame(checks)
display(quality_report)
print('Checks failing:', int((quality_report['status'] == 'FAIL').sum()), 'of', len(quality_report))


,dimension,column,issue,failed_count,total_count,status,expected_rule,recommended_action
0,Completeness,order_id,Required value missing,0,12,PASS,Required fields must be populated,Recover from source; do not invent values.
1,Completeness,order_date,Required value missing,1,12,FAIL,Required fields must be populated,Recover from source; do not invent values.
2,Completeness,customer_segment,Required value missing,0,12,PASS,Required fields must be populated,Recover from source; do not invent values.
3,Completeness,city,Required value missing,1,12,FAIL,Required fields must be populated,Recover from source; do not invent values.
4,Completeness,category,Required value missing,0,12,PASS,Required fields must be populated,Recover from source; do not invent values.
5,Completeness,quantity,Required value missing,0,12,PASS,Required fields must be populated,Recover from source; do not invent values.
6,Completeness,unit_price,Required value missing,0,12,PASS,Required fields must be populated,Recover from source; do not invent values.
7,Completeness,payment_status,Required value missing,0,12,PASS,Required fields must be populated,Recover from source; do not invent values.
8,Completeness,discount_pct,Discount missing,1,12,FAIL,Missing discount needs business justification,Do not assume zero without approval.
9,Uniqueness,order_id,Order ID duplicated or missing,2,12,FAIL,Order ID required and unique,Investigate repeated identifiers with the sour...


Checks failing: 7 of 17


## 4. Create a review copy (without silently changing the raw data)
This creates normalized helper columns for analysis. It does not delete duplicate records or fill missing values. Rows failing critical checks should be reviewed before official reporting.

In [5]:
df = df_raw.copy()
df['order_date_parsed'] = parsed_date
df['quantity_numeric'] = quantity_num
df['unit_price_numeric'] = price_num
df['discount_numeric'] = discount_num
df['customer_segment_normalized'] = df['customer_segment'].astype('string').str.strip().str.title()
df['category_normalized'] = df['category'].astype('string').str.strip().str.title()
df['payment_status_normalized'] = df['payment_status'].astype('string').str.strip().str.title()
display(df.head())


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status,order_date_parsed,quantity_numeric,unit_price_numeric,discount_numeric,customer_segment_normalized,category_normalized,payment_status_normalized
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid,2026-01-03,2,799,10.0,Student,Learning Kit,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid,NaT,1,1499,0.0,Fresher,Course Access,Paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending,2026-01-05,1,1499,NaN,Student,Course Access,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid,2026-01-07,3,799,5.0,Professional,Learning Kit,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid,2026-01-07,3,799,5.0,Professional,Learning Kit,Paid


## 5. KPI calculation (provisional)
Because this sample contains data-quality failures, the figures below are provisional. The code excludes records with critical issues and duplicate order IDs; confirm the business rules with the KPI owner before publishing.

In [6]:
critical_bad = bad_date | bad_qty | bad_price | bad_discount | dup_order_mask | df_raw['order_id'].isna()
valid = df.loc[~critical_bad].copy()
valid['gross_sales'] = valid['quantity_numeric'] * valid['unit_price_numeric']
valid['net_sales'] = valid['gross_sales'] * (1 - valid['discount_numeric'] / 100)
order_count = valid['order_id'].nunique()
units = valid['quantity_numeric'].sum()
net_sales = valid['net_sales'].sum()
status_counts = valid.drop_duplicates('order_id')['payment_status_normalized'].value_counts()
status_denominator = valid['order_id'].nunique()
kpi_summary = pd.DataFrame([
    ['Gross Sales (provisional)', valid['gross_sales'].sum()],
    ['Net Sales After Discount (provisional)', net_sales],
    ['Unique Orders (provisional)', order_count],
    ['Units Sold (provisional)', units],
    ['Average Order Value (provisional)', net_sales / order_count if order_count else np.nan],
    ['Average Selling Price per Unit (provisional)', net_sales / units if units else np.nan],
    ['Paid Order Rate (%)', status_counts.get('Paid', 0) / status_denominator * 100 if status_denominator else np.nan],
    ['Pending Payment Rate (%)', status_counts.get('Pending', 0) / status_denominator * 100 if status_denominator else np.nan],
    ['Failed Payment Rate (%)', status_counts.get('Failed', 0) / status_denominator * 100 if status_denominator else np.nan],
    ['Refunded Order Rate (%)', status_counts.get('Refunded', 0) / status_denominator * 100 if status_denominator else np.nan],
], columns=['KPI', 'Value'])
display(kpi_summary)
print(f'Records excluded from provisional KPI calculation: {int(critical_bad.sum())} of {len(df_raw)}')


,KPI,Value
0,Gross Sales (provisional),6394.000000
1,Net Sales After Discount (provisional),5784.500000
2,Unique Orders (provisional),4.000000
3,Units Sold (provisional),6.000000
4,Average Order Value (provisional),1446.125000
5,Average Selling Price per Unit (provisional),964.083333
6,Paid Order Rate (%),50.000000
7,Pending Payment Rate (%),0.000000
8,Failed Payment Rate (%),25.000000
9,Refunded Order Rate (%),25.000000


Records excluded from provisional KPI calculation: 8 of 12


## 6. Export evidence for submission
Exports the executable check results and a column-level profile. Keep these outputs with the notebook as evidence.

In [8]:
profile.to_csv('retail_data_profile.csv', index=False)
quality_report.to_csv('retail_quality_check_results.csv', index=False)
kpi_summary.to_csv('retail_kpi_summary_provisional.csv', index=False)
print('Created: retail_data_profile.csv, retail_quality_check_results.csv, retail_kpi_summary_provisional.csv')
# In Google Colab, uncomment to download each output:
# from google.colab import files
# files.download('retail_data_profile.csv')
# files.download('retail_quality_check_results.csv')
# files.download('retail_kpi_summary_provisional.csv')


Created: retail_data_profile.csv, retail_quality_check_results.csv, retail_kpi_summary_provisional.csv


In [9]:
from google.colab import files

files.download('retail_data_profile.csv')
files.download('retail_quality_check_results.csv')
files.download('retail_kpi_summary_provisional.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>